# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is FAIR (Findable, Accessible, Interoperable, and Reusable) ready. This notebook demonstrates how to:
- Load dataset metadata and records from the Croissant schema
- Explore record sets, fields, and column IDs (always using `@id`)
- Extract specific data into DataFrames
- Conduct simple exploratory data analysis
- Visualize data distributions


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset's Croissant schema
dataset = mlc.Dataset(croissant_url)

# Get the dataset metadata as a plain dict
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n{'=' * len(metadata['name'])}\n{metadata['description']}")

## 2. Data Overview
Review record set `@id`s, their fields and field `@id`s as defined in the Croissant schema.

In [ ]:
# List available RecordSets and show their Fields, all by @id
record_sets = dataset.record_sets  # List of RecordSet objects
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"  Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} ({field.name})")
    print('-'*50)
# Store the RecordSet @id(s) and some example Field @ids for use in the next steps
if record_sets:
    first_record_set_id = record_sets[0].id
    example_field_id = record_sets[0].fields[0].id if record_sets[0].fields else None


## 3. Data Extraction
Load data from each record set into pandas DataFrames. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data into DataFrames for all available RecordSets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id} (shape: {df.shape})")
    else:
        print(f"No records found for RecordSet @id: {rs_id}")

# Display columns of the first DataFrame (by RecordSet @id)
main_rs = first_record_set_id
if main_rs in dataframes:
    print(f"\nColumns for RecordSet {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records on a selected numeric field, normalize, and group by another field. All columns/fields are referenced by their `@id`.

In [ ]:
# Identify a numeric field and a suitable group field (using @id from earlier inspection)
df = dataframes[main_rs]
print("Sample columns (by @id):", df.columns.tolist())

# Attempt to auto-select a numeric column and a grouping column by inspecting dtypes
numeric_field_id = None
group_field_id = None
for c in df.columns:
    try:
        # Try to convert to numeric
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
        elif pd.to_numeric(df[c].dropna().iloc[:10], errors='coerce').notnull().any():
            numeric_field_id = c
            break
    except Exception:
        continue
for c in df.columns:
    if c != numeric_field_id and df[c].nunique() < min(5, len(df)//2):
        group_field_id = c
        break
print(f"Numeric field @id selected: {numeric_field_id}")
print(f"Group field @id selected: {group_field_id}")

# Convert numeric field to float if not already
df_numeric = df.copy()
df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')
threshold = df_numeric[numeric_field_id].mean() if pd.notnull(df_numeric[numeric_field_id].mean()) else 0
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (threshold=mean):")
display(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (z-score)
f_norm = f"{numeric_field_id}_normalized"
filtered_df[f_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f_norm]].head())

# Group by the selected group_field_id, if any
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id} per group):")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and compare across groups if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df_numeric[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field grouped by group_field_id, if applicable
if group_field_id in df_numeric.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_numeric)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR dataset using the Croissant schema and the `mlcroissant` library. We:
- Accessed dataset metadata and structure via the Croissant URL.
- Inspected record sets and referenced all entities via their `@id` fields for precise tracking.
- Extracted tabular data and conducted basic filtering, normalization, grouping, and visualization.

This approach ensures interoperability and traceability of data elements, and prepares the dataset for advanced analytics or machine learning workflows.

---

*For more information on Croissant or the FAIR² dataset, visit [https://sen.science/doi/10.71728/senscience.qs2f-h81p](https://sen.science/doi/10.71728/senscience.qs2f-h81p)*